# 02 - Text Preprocessing

Clean and normalize the raw tweets before feeding them to TF-IDF. We show each step and its effect on a few examples.

**IMPORTANT:** We do *not* aggressively remove stop words, because negation words like `not`, `never`, `no` carry sentiment. Removing them would turn `"not good"` into `"good"`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.preprocessing import clean_text, remove_stop_words, preprocess_text

In [ ]:
examples = [
    "@VirginAmerica the app is VERY slow!!! https://t.co/abc #help",
    "Payment failed during checkout &amp; I got charged twice!!!",
    "This is NOT good. Never using this again.",
]

for ex in examples:
    print("RAW: ", ex)
    print("CLEAN:", clean_text(ex))
    print()

## What `clean_text` does

- Lowercases
- Removes URLs, emails, HTML tags, @handles, hashtag symbols
- Keeps letters, numbers, and basic punctuation (punctuation can matter)
- Collapses whitespace

## Stop words and negation

We keep a custom stop-word list that **excludes negation words**.

In [ ]:
tests = [
    "this is not a good product",
    "I never received my refund",
    "the payment gateway failed",
]
for t in tests:
    print(f"INPUT : {t}")
    print(f"STOPS : {remove_stop_words(t)}")
    print()

Notice how `not` and `never` survive stop-word removal. This is critical for sentiment.

## Full pipeline on the dataset

We apply `clean_text` (no lemmatization by default to keep this simple and fast). The cleaned text is what TF-IDF will use.

In [ ]:
from src.data_loader import load_raw_tweets

df = load_raw_tweets()
df["clean_text"] = df["text"].apply(
    lambda x: preprocess_text(x, clean=True, remove_stops=False, lemmatize=False)
)

pd.set_option("display.max_colwidth", 100)
df[["text", "clean_text"]].sample(5, random_state=42)

In [ ]:
empty = (df["clean_text"].str.len() == 0).sum()
print(f"Rows with empty cleaned text: {empty}")
print(f"Original rows: {len(df)}")

## Tokenization

Tokenization splits text into individual tokens. It happens automatically inside scikit-learn's `TfidfVectorizer`, so we usually don't need to do it by hand — but here is what it does.

In [ ]:
from src.preprocessing import tokenize_text

print(tokenize_text("payment failed during checkout"))

## Lemmatization (optional)

Lemmatization reduces words to their base form (`running` -> `run`, `feet` -> `foot`). It requires spaCy:

```
pip install spacy
python -m spacy download en_core_web_sm
```

It is optional here; the classical models work fine without it (TF-IDF already handles word variants statistically).

## Summary

Preprocessing is done. Next we build the classic TF-IDF + Logistic Regression sentiment model in `03_tfidf_sentiment.ipynb`.